In [4]:

import re

def parse(s):
    tokens = re.findall(r'[A-Za-z_]\w*|\d+|[(),]', s)
    pos = [0]

    def term():
        name = tokens[pos[0]]
        pos[0] += 1

        if pos[0] < len(tokens) and tokens[pos[0]] == '(':
            pos[0] += 1
            args = []
            while True:
                args.append(term())
                if tokens[pos[0]] == ')':
                    pos[0] += 1
                    break
                pos[0] += 1
            return (name, args)

        return name

    return term()


def is_var(x):
    return isinstance(x, str) and (
        x[0].isupper() or x in ('x', 'y', 'z')
    )


def apply(x, subs):
    while isinstance(x, str) and x in subs:
        x = subs[x]
    if isinstance(x, tuple):
        return (x[0], [apply(a, subs) for a in x[1]])
    return x


def unify(x, y, subs):
    x = apply(x, subs)
    y = apply(y, subs)

    if x == y:
        return True

    if is_var(x):
        if occurs(x, y, subs):
            return False
        subs[x] = y
        return True

    if is_var(y):
        if occurs(y, x, subs):
            return False
        subs[y] = x
        return True

    if isinstance(x, tuple) and isinstance(y, tuple):
        if x[0] != y[0] or len(x[1]) != len(y[1]):
            return False

        for a, b in zip(x[1], y[1]):
            if not unify(a, b, subs):
                return False
        return True

    return False


def occurs(var, term, subs):
    term = apply(term, subs)

    if var == term:
        return True

    if isinstance(term, tuple):
        return any(occurs(var, a, subs) for a in term[1])

    return False


while True:
    first = input("\nEnter first statement: ")

    if first.lower() == "exit":
        print("Program terminated.")
        break

    second = input("Enter second statement: ")

    try:
        subs = {}

        if unify(parse(first), parse(second), subs):
            print("Unification Successful")
            print("Substitutions:", subs)
        else:
            print("Not Unified")

    except:
        print("Invalid Input")



Enter first statement: p(f(a),g(Y))
Enter second statement: p(X,X)
Not Unified

Enter first statement: prime(11)
Enter second statement: prime(y)
Unification Successful
Substitutions: {'y': '11'}

Enter first statement: Q(a,g(x,a),f(y))
Enter second statement: Q(a,g(f(b),a),x)
Unification Successful
Substitutions: {'x': ('f', ['b']), 'y': 'b'}

Enter first statement: p(b,X,f(g(Z)))
Enter second statement: p(Z,f(Y),f(Y))
Unification Successful
Substitutions: {'Z': 'b', 'X': ('f', ['Y']), 'Y': ('g', ['b'])}

Enter first statement: exit
Program terminated.
